# Episode 007: long-integration limit-cycle diagnostics

This clean-run notebook is the authoritative numerical reference for the high-aerosol Figure 4 center case. It integrates the no-evaporation model in `(log(n), log(q), s)` coordinates with adaptive Dormand--Prince RK45, assesses 300 linearized periods, and regenerates the curated figures and browser-validation fixtures in `outputs/`.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

from bergner_spichtinger_2026.constants import Environment
from bergner_spichtinger_2026.core import equilibrium, process_terms, vector_field
from bergner_spichtinger_2026.limit_cycles import analyze_late_cycle_drift, extract_cycles, phase_independent_orbit_distance
from bergner_spichtinger_2026.stability import physical_eigenvalues

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'pyproject.toml').is_file():
    REPO_ROOT = Path.cwd().resolve().parents[2]
OUTPUTS = REPO_ROOT / 'episodes/007-limit-cycle-interactive-widget/outputs'
OUTPUTS.mkdir(parents=True, exist_ok=True)

env = Environment(p=30000.0, T=225.0, w=0.1, F=1.0, N_a=1.0e10, Δz=100.0, include_evaporation=False)
solver_settings = {'method': 'RK45', 'coordinates': ['log(n)', 'log(q)', 's'], 'rtol': 1e-8, 'atol': 1e-10, 'max_step_fraction_of_linear_period': 1 / 15}
x_eq = equilibrium(env)
eigenvalues = physical_eigenvalues(x_eq, env=env)
linear_period = float(2 * np.pi / np.max(eigenvalues.imag))
horizon_periods = 300
horizon = horizon_periods * linear_period
x_eq, eigenvalues, linear_period, horizon


(array([3.88673905e+04, 5.26714066e-06, 1.46824610e+00]),
 array([ 0.00021372+0.00788295j,  0.00021372-0.00788295j,
        -0.00506165+0.j        ]),
 797.0601943441069,
 239118.05830323207)

In [2]:
def log_rhs(_time, log_state):
    n, q = np.exp(log_state[:2])
    tendency = vector_field(n, q, log_state[2], env)
    return np.array([tendency[0] / n, tendency[1] / q, tendency[2]])

starts = {
    'paper_0.99': 0.99 * x_eq,
    'n_plus_1pct': x_eq * np.array([1.01, 1.0, 1.0]),
    'q_plus_1pct': x_eq * np.array([1.0, 1.01, 1.0]),
    's_plus_1pct': x_eq * np.array([1.0, 1.0, 1.01]),
}

def integrate(start):
    initial = np.array([np.log(start[0]), np.log(start[1]), start[2]])
    solution = solve_ivp(log_rhs, (0.0, horizon), initial, method='RK45', rtol=solver_settings['rtol'], atol=solver_settings['atol'], max_step=linear_period / 15)
    if not solution.success:
        raise RuntimeError(solution.message)
    physical = np.column_stack((np.exp(solution.y[0]), np.exp(solution.y[1]), solution.y[2]))
    return solution.t, physical

trajectories = {name: integrate(start) for name, start in starts.items()}
cycle_extractions = {name: extract_cycles(time, state[:, 2]) for name, (time, state) in trajectories.items()}
drift = {name: analyze_late_cycle_drift(extraction.cycles, window=20, threshold=1e-3) for name, extraction in cycle_extractions.items()}
[(name, len(cycle_extractions[name].cycles), drift[name].converged) for name in starts]


[('paper_0.99', 100, True),
 ('n_plus_1pct', 115, True),
 ('q_plus_1pct', 119, True),
 ('s_plus_1pct', 95, True)]

In [3]:
def complete_cycle_samples(time, state, cycle):
    mask = (time >= cycle.start.time) & (time <= cycle.end.time)
    selected_time, selected_state = time[mask], state[mask]
    if selected_time[0] > cycle.start.time:
        selected_time = np.insert(selected_time, 0, cycle.start.time)
        selected_state = np.vstack((np.array([np.interp(cycle.start.time, time, state[:, j]) for j in range(3)]), selected_state))
    if selected_time[-1] < cycle.end.time:
        selected_time = np.append(selected_time, cycle.end.time)
        selected_state = np.vstack((selected_state, np.array([np.interp(cycle.end.time, time, state[:, j]) for j in range(3)])))
    return selected_time, selected_state

reference_name = 'paper_0.99'
reference_time, reference_state = trajectories[reference_name]
reference_cycle = cycle_extractions[reference_name].cycles[-1]
reference_orbit = complete_cycle_samples(reference_time, reference_state, reference_cycle)
orbit_distances = {}
for name, (time, state) in trajectories.items():
    candidate_orbit = complete_cycle_samples(time, state, cycle_extractions[name].cycles[-1])
    distance = phase_independent_orbit_distance(*reference_orbit, *candidate_orbit, samples=512)
    orbit_distances[name] = {'distance': distance.distance, 'phase_shift_cycles': distance.phase_shift, 'samples': distance.samples}
converged = all(item.converged for item in drift.values()) and all(item['distance'] <= 1e-3 for item in orbit_distances.values())
converged, orbit_distances


(True,
 {'paper_0.99': {'distance': 0.0,
   'phase_shift_cycles': 1.3552527156068805e-18,
   'samples': 512},
  'n_plus_1pct': {'distance': 0.0001548267644586064,
   'phase_shift_cycles': 2.492689789521248e-06,
   'samples': 512},
  'q_plus_1pct': {'distance': 9.246478254969989e-05,
   'phase_shift_cycles': 5.62480677673206e-07,
   'samples': 512},
  's_plus_1pct': {'distance': 0.00012453331194047356,
   'phase_shift_cycles': 0.9999995803142709,
   'samples': 512}})

In [4]:
# Long-run stability and all-start attractor-convergence figures.
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True, constrained_layout=True)
for name, extraction in cycle_extractions.items():
    cycles = extraction.cycles
    axes[0].plot(range(1, len(cycles) + 1), [cycle.period for cycle in cycles], label=name)
    axes[1].plot(range(1, len(cycles) + 1), [cycle.amplitude for cycle in cycles], label=name)
axes[0].set(ylabel='period [s]', title='Limit-cycle stability over 300 linearized periods')
axes[1].set(xlabel='complete-cycle index', ylabel='s peak-to-peak amplitude')
axes[0].legend(ncol=2, fontsize=8)
fig.savefig(OUTPUTS / 'limit_cycle_stability.png', dpi=200)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
for name, (time, state) in trajectories.items():
    ax.plot(np.log10(state[:, 0]), state[:, 2], lw=0.75, label=name)
ax.set(xlabel='log10(n) [log10(kg dry-air^-1)]', ylabel='s', title='Attraction to the common limit cycle')
ax.legend(fontsize=8)
fig.savefig(OUTPUTS / 'attractor_convergence_log10n_s.png', dpi=200)
plt.close(fig)


In [5]:
# Representative final cycle, with all required process budgets and total tendencies.
time, state = trajectories[reference_name]
cycle = cycle_extractions[reference_name].cycles[-1]
cycle_time, cycle_state = complete_cycle_samples(time, state, cycle)
relative_time = cycle_time - cycle_time[0]
terms = {key: np.array([process_terms(*row, env)[key] for row in cycle_state]) for key in process_terms(*cycle_state[0], env)}
totals = np.array([vector_field(*row, env) for row in cycle_state])
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True, constrained_layout=True)
axes[0].plot(relative_time, cycle_state[:, 0], label='n'); axes[0].plot(relative_time, cycle_state[:, 1], label='q'); axes[0].plot(relative_time, cycle_state[:, 2], label='s'); axes[0].set(ylabel='state', title='One-cycle state and process budgets'); axes[0].legend(ncol=3)
for label in ('Nuc_n', 'Sed_n'): axes[1].plot(relative_time, terms[label], label=label)
axes[1].plot(relative_time, totals[:, 0], color='black', lw=1.5, label='total dn/dt'); axes[1].set(ylabel='dn/dt'); axes[1].legend(ncol=3)
for label in ('Nuc_q', 'Dep_q', 'Sed_q'): axes[2].plot(relative_time, terms[label], label=label)
axes[2].plot(relative_time, totals[:, 1], color='black', lw=1.5, label='total dq/dt'); axes[2].set(ylabel='dq/dt'); axes[2].legend(ncol=4)
for label in ('Cool', 'Nuc_s', 'Dep_s'): axes[3].plot(relative_time, terms[label], label=label)
axes[3].plot(relative_time, totals[:, 2], color='black', lw=1.5, label='total ds/dt'); axes[3].set(xlabel='time since cycle start [s]', ylabel='ds/dt'); axes[3].legend(ncol=4)
fig.savefig(OUTPUTS / 'one_cycle_state_process_budgets.png', dpi=200)
plt.close(fig)


In [6]:
# Reference CSVs and schema-versioned metadata for browser validation.
summary_rows = []
for name, extraction in cycle_extractions.items():
    for index, item in enumerate(extraction.cycles):
        previous = extraction.cycles[index - 1] if index else None
        summary_rows.append({'start': name, 'cycle_index': index + 1, 'start_time_s': item.start.time, 'end_time_s': item.end.time, 'period_s': item.period, 's_max': item.maximum.value, 's_min': item.minimum.value, 's_amplitude': item.amplitude, 'period_relative_drift': np.nan if previous is None else abs(item.period - previous.period) / previous.period, 'amplitude_relative_drift': np.nan if previous is None else abs(item.amplitude - previous.amplitude) / previous.amplitude})
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUTS / 'per_cycle_summary.csv', index=False, float_format='%.17g')

early_end = min(2 * linear_period, time[-1])
final_start = cycle_extractions[reference_name].cycles[-3].start.time
final_end = cycle_extractions[reference_name].cycles[-1].end.time
selection = (time <= early_end) | ((time >= final_start) & (time <= final_end))
reference = pd.DataFrame({'time_s': time[selection], 'n_kg_dry_air_minus1': state[selection, 0], 'q_kg_kg_dry_air_minus1': state[selection, 1], 's': state[selection, 2]})
final_boundary = {'time_s': final_end, 'n_kg_dry_air_minus1': np.interp(final_end, time, state[:, 0]), 'q_kg_kg_dry_air_minus1': np.interp(final_end, time, state[:, 1]), 's': np.interp(final_end, time, state[:, 2])}
reference = pd.concat((reference, pd.DataFrame([final_boundary])), ignore_index=True).drop_duplicates(subset='time_s')
reference.to_csv(OUTPUTS / 'reference_trajectory.csv', index=False, float_format='%.17g')

metadata = {'schema_version': '1.0.0', 'canonical_parameters': {'T': 225.0, 'p': 30000.0, 'w': 0.1, 'F': 1.0, 'N_a': 1.0e10, 'Delta_z': 100.0, 'include_evaporation': False}, 'units': {'T': 'K', 'p': 'Pa', 'w': 'm s^-1', 'F': '1', 'N_a': 'm^-3', 'Delta_z': 'm', 'n': 'kg dry-air^-1', 'q': 'kg kg_dry-air^-1', 's': '1', 'time': 's'}, 'equilibrium': {'n': x_eq[0], 'q': x_eq[1], 's': x_eq[2]}, 'linearized_eigenvalues_per_s': [{'real': value.real, 'imag': value.imag} for value in eigenvalues], 'linearized_period_s': linear_period, 'integration_horizon': {'periods': horizon_periods, 'seconds': horizon}, 'initial_conditions': {name: {'n': value[0], 'q': value[1], 's': value[2]} for name, value in starts.items()}, 'solver_settings': solver_settings, 'cycle_boundaries': {name: [{'start_s': item.start.time, 'end_s': item.end.time} for item in extraction.cycles] for name, extraction in cycle_extractions.items()}, 'convergence': {'final_window_cycles': 20, 'drift_threshold': 1e-3, 'per_start': {name: {'period_drifts': value.period_drifts.tolist(), 'amplitude_drifts': value.amplitude_drifts.tolist(), 'passed': value.converged} for name, value in drift.items()}, 'orbit_distance_threshold': 1e-3, 'orbit_distances': orbit_distances, 'passed': converged}, 'orbit_metric': {'function': 'phase_independent_orbit_distance', 'states': ['n', 'q', 's'], 'samples': 512, 'definition': 'symmetric normalized RMS distance after cyclic phase-shift minimization; each component is scaled by its combined peak-to-peak range with a scale floor'}, 'reference_provenance': {'notebook': 'notebooks/01_limit_cycle_diagnostics.ipynb', 'trajectory_selection': 'first two linearized periods plus final three complete cycles', 'csv_float_format': '%.17g', 'figures': ['limit_cycle_stability.png', 'attractor_convergence_log10n_s.png', 'one_cycle_state_process_budgets.png']}}
(OUTPUTS / 'reference_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')
assert converged, 'Late-cycle drift or all-start orbit convergence did not satisfy the approved threshold.'
print(f'Regenerated outputs in {OUTPUTS}; convergence passed: {converged}')


Regenerated outputs in /home/iross/research/ongoing/ai-class/code/bergner-2026/episodes/007-limit-cycle-interactive-widget/outputs; convergence passed: True
